# Step 2: Data extraction and ticker handling

The goal of this notebook is one dataset of Trump's equity trades in 2025 and 2026 where every trade has a standardized ticker. The logic flows as follows:

1. Ensuring that the 2026 dataset of tickers is complete
2. Adding tickers to the 2025 dataset
3. Merging the datasets
4. Handling and standardizing different ticker symbols for a given corporation
5. Ticker corrections after price fetching: the errors only visible once a ticker meets its price series
6. The purchase table for the price step

Sections 1–4 give the cleaned file (kept as the sheet `TradesBeforePriceReview`); sections 5–6 give the final file (sheet `Trades`) and the purchase table that Step 3 prices. Every manual decision sits in `TickerReview.xlsx`, one sheet per review, in the order of the sections; the output is `Trump_trades_2025_2026.xlsx`.


In [1]:
import pandas as pd
import numpy as np
import os

excel_path = "TickerReview.xlsx"

pd.set_option('future.no_silent_downcasting', True)

C:\Users\andre\AppData\Local\Temp\ipykernel_18084\1949819719.py:7: Pandas4Warning: 'future.no_silent_downcasting' is deprecated, please refrain from using it.
  pd.set_option('future.no_silent_downcasting', True)


## 1) Data import and ensuring that the 2026 dataset of tickers is complete

We import the raw 2025 and 2026 disclosures, align the two datasets and make sure that every equity investment in 2026 has a ticker beside it.

In [2]:
trump2025 = pd.read_csv("RawData/Trump_trades_2025_RAW.csv", sep=";")
trump2026 = pd.read_csv("RawData/Trump_trades_2026_RAW (quiver).csv", sep=";")
display(trump2025.head(5))
display(trump2026.head(5))

,description,type,date,amount_range,amount_low_usd,amount_high_usd
0,ELEVANCE HEALTH INC,purchase,06.01.2025,"$15,001 - $50,000",15001,50000
1,CONSTELLATION BRANDS INC,purchase,06.01.2025,"$15,001 - $50,000",15001,50000
2,BIOGEN INC,purchase,06.01.2025,"$15,001 - $50,000",15001,50000
3,UNITED PARCEL SVC INC,purchase,06.01.2025,"$15,001 - $50,000",15001,50000
4,UBER TECHNOLOGIES INC,purchase,06.01.2025,"$15,001 - $50,000",15001,50000


,Ticker,Company,Transaction,Amount,Filed,Traded,ExcessReturn
0,TT,TRANE TECHNOLOGIES PLC F,Purchase,"$1,001 - $15,000",13.05.2026,02.01.2026,0.701524
1,SNDK,SANDISK CORP COM,Purchase,"$1,001 - $15,000",13.05.2026,06.01.2026,374.472968
2,AMD,ADVANCED MICRO DEVICES INC UNSOLICITED,Purchase,"$50,001 - $100,000",13.05.2026,06.01.2026,170.702536
3,HPE,Hewlett Packard Enterprise Co,Purchase,"$100,001 - $250,000",13.05.2026,06.01.2026,147.357128
4,FTNT,FORTINET INC,Sale,"$250,001 - $500,000",13.05.2026,06.01.2026,114.353350


### 1.1) Standardizing the data

* Standardizing column names for both datasets
* Dropping unnecessary columns
* Converting column types

In [3]:
# Drop unnecessary columns from the 2026 dataset
trump2026.drop(columns=["Filed", "ExcessReturn"], inplace=True)

# changing and splitting the Amount column into two new columns: amount_low and amount_high (matching the 2025 naming convention)
trump2026[["amount_low", "amount_high"]] = (
    trump2026["Amount"]
    .str.replace("$", "", regex=False)
    .str.replace(",", "", regex=False)
    .str.split(" - ", expand=True)
    .astype(int)
)
trump2026.drop(columns=["Amount"], inplace=True)

# Converting Traded to an actual date column (some rows carry a trailing " 00:00" time, stripped before parsing)
trump2026["Traded"] = pd.to_datetime(trump2026["Traded"].str.split(" ").str[0], format="%d.%m.%Y")

In [4]:
# Renaming columns in the 2025 dataset to match the 2026 dataset
trump2025.rename(columns={"amount_low_usd": "amount_low", 
    "amount_high_usd": "amount_high", 
    "description": "Company", 
    "date": "Traded",
    "type": "Transaction",
    }, 
    inplace=True)
# Dropping the "amount_range" column from the 2025 dataset as it is not needed
trump2025.drop(columns=["amount_range"], inplace=True)

# Converting Traded to an actual date column
trump2025["Traded"] = pd.to_datetime(trump2025["Traded"], format="%d.%m.%Y")

In [5]:
display(trump2025.head())
display(trump2026.head())

,Company,Transaction,Traded,amount_low,amount_high
0,ELEVANCE HEALTH INC,purchase,2025-01-06,15001,50000
1,CONSTELLATION BRANDS INC,purchase,2025-01-06,15001,50000
2,BIOGEN INC,purchase,2025-01-06,15001,50000
3,UNITED PARCEL SVC INC,purchase,2025-01-06,15001,50000
4,UBER TECHNOLOGIES INC,purchase,2025-01-06,15001,50000


,Ticker,Company,Transaction,Traded,amount_low,amount_high
0,TT,TRANE TECHNOLOGIES PLC F,Purchase,2026-01-02,1001,15000
1,SNDK,SANDISK CORP COM,Purchase,2026-01-06,1001,15000
2,AMD,ADVANCED MICRO DEVICES INC UNSOLICITED,Purchase,2026-01-06,50001,100000
3,HPE,Hewlett Packard Enterprise Co,Purchase,2026-01-06,100001,250000
4,FTNT,FORTINET INC,Sale,2026-01-06,250001,500000


### 1.2) Ensuring that the 2026 dataset of tickers is complete

The 2026 dataset from Quiver Quant comes with a ticker column, but not all rows are filled in. We complete it in three steps:

1) Backfill/frontfill missing tickers in 2026
2) Removing non-equities from the list
3) Accounting for naming discrepancies in (1)

**Stock classes:** the 2026 data does not distinguish consistently between share classes. It generally uses the most common ticker for each company. Google, for instance, is represented by GOOG (Class C), even when the company name refers to Class A shares (GOOGL). We do not consider this distinction material for our analysis. The share classes differ in voting rights, share splits and other corporate characteristics, but they represent the same underlying company and generally follow the same market value. We standardize these cases in section 4.

#### 1.2.1) Backfill/frontfill missing tickers in 2026

For companies that appear several times, we take the ticker from one observation and fill it into the other observations of the same company, both forward (from an earlier row) and backward (from a later row).

Reasoning: we need to ensure that all of the rows actually have a corresponding ticker beside them, so we frontfill/backfill for all companies. Despite many investments being in bonds, ETFs, etc., there are instances where equities actually have missing tickers.

In [6]:
# Fill missing tickers using other rows that share the same Company name, via forward fill and backward fill
trump2026["Ticker"] = trump2026.groupby("Company")["Ticker"].transform(lambda s: s.ffill().bfill())

still_missing_2026 = trump2026["Ticker"].isna().sum()
print(f"{still_missing_2026} rows still missing a ticker after within-company fill")

321 rows still missing a ticker after within-company fill


#### 1.2.2) Removing non-equities from the list

Since we only focus on equities, we remove all investments that are not equities. We apply a regex pattern on the company name to flag indices, bonds, notes, money-market funds and ETFs, for both the 2025 and the 2026 dataset. For 2026 we also clear any ticker that sits on a flagged row, including tickers we just filled in above or that came from Quiver, since a fund's own ticker is not a stock ticker. We export the flagged rows to `TickerReview.xlsx` and go over the list manually to ensure that only non-equities are removed. The flag is used for every missing-ticker count, matching step and manual review from here on.

Reasoning: we have observed some instances where indices and bonds have an associated ticker.

In [7]:
import re

NON_EQUITY_PATTERN = re.compile(
    r"\bDUE\b|\bCNTY\b|\bSCH\s?DIST\b|\bISD\b|\bTWP\b|\bB/E\b|\bPROM\s?NT\b|\bRFDG\b|\bMUNI\b"
    r"|\bREV(?:ENUE)?\s+BOND\b|\bSER\s+[0-9A-Z]{1,4}\b|%|\bFUND\b|\bMONEY\s?(?:MKT|MARKET)\b|\bETF\b"
    r"|\bFEDERATED\b|\bCASH\b|\bRESERVES\b |\bISHARES\b"
    r"|,\s*(?:19|20)\d{2}\b"
    r"|\d{1,2}/\d{1,2}/\d{2,4}",      # bond maturity date, e.g. "12/29/49"
    re.IGNORECASE,
)

trump2026["IsNonEquity"] = trump2026["Company"].str.contains(NON_EQUITY_PATTERN, regex=True, na=False)
trump2025["IsNonEquity"] = trump2025["Company"].str.contains(NON_EQUITY_PATTERN, regex=True, na=False)

# Clear any ticker already assigned to a non-equity row -- even one just filled in by the
# within-company fill above, or provided directly by Quiver (e.g. a money-market fund's own ticker).
cleared_2026 = trump2026.loc[trump2026["IsNonEquity"], "Ticker"].notna().sum()
trump2026.loc[trump2026["IsNonEquity"], "Ticker"] = pd.NA

print(f"2026: flagged as non-equity: {trump2026['IsNonEquity'].sum()} rows, "
      f"{trump2026[trump2026['IsNonEquity']]['Company'].nunique()} distinct companies "
      f"({cleared_2026} of those rows had a ticker cleared)")
print(f"2025: flagged as non-equity: {trump2025['IsNonEquity'].sum()} rows, "
      f"{trump2025[trump2025['IsNonEquity']]['Company'].nunique()} distinct companies")

2026: flagged as non-equity: 330 rows, 258 distinct companies (53 of those rows had a ticker cleared)
2025: flagged as non-equity: 183 rows, 50 distinct companies


In [8]:
NonEquities2026 = trump2026[trump2026["IsNonEquity"]==True]
# sending to excel for manual review - only the "2026NonEquities" sheet, without touching any other sheets in the workbook
if os.path.exists(excel_path):
    with pd.ExcelWriter(excel_path, engine="openpyxl", mode="a", if_sheet_exists="replace") as writer:
        NonEquities2026.to_excel(writer, sheet_name="2026NonEquities")
else:
    NonEquities2026.to_excel(excel_path, sheet_name="2026NonEquities")

NonEquities2025 = trump2025[trump2025["IsNonEquity"]==True]
# sending to excel for manual review - only the "2025NonEquities" sheet, without touching any other sheets in the workbook
if os.path.exists(excel_path):
    with pd.ExcelWriter(excel_path, engine="openpyxl", mode="a", if_sheet_exists="replace") as writer:
        NonEquities2025.to_excel(writer, sheet_name="2025NonEquities")
else:
    NonEquities2025.to_excel(excel_path, sheet_name="2025NonEquities")


In [9]:
# Overview of companies with NO ticker anywhere in the 2026 dataset (ffill/bfill already resolved the rest).
# Non-equity holdings are excluded here too -- no point manually reviewing a bond/fund for a "real" ticker.
missing_tickers_2026 = (
    trump2026[~trump2026["IsNonEquity"]].groupby('Company')['Ticker']
    .agg(missing=lambda s: s.isna().sum(), present=lambda s: s.notna().sum())
)
missing_tickers_2026 = missing_tickers_2026[missing_tickers_2026['missing'] > 0].sort_values('missing', ascending=False)
display(missing_tickers_2026)

# sending to excel for manual review - only the "2026MissingTickers" sheet, without touching any other sheets in the workbook
if os.path.exists(excel_path):
    with pd.ExcelWriter(excel_path, engine="openpyxl", mode="a", if_sheet_exists="replace") as writer:
        missing_tickers_2026.to_excel(writer, sheet_name="2026MissingTickers")
else:
    missing_tickers_2026.to_excel(excel_path, sheet_name="2026MissingTickers")

,missing,present
Company,,
EVEREST REINSURANCE GR F,3,0
INVESCO QQQ TR,3,0
VANGUARD INTERMEDIATE-TERM CORPORATE BOND INDEX FUN,3,0
Invesco Exchange-traded Fd Tr S&p 500 Equal Weight,2,0
FRONTIER COMMUNICATIONS,2,0
CARNIVAL CORP PAIRED CTF 1 COM CARNI,2,0
ERIE INDTV CO CLASS A,2,0
ADTALEN GLOBAL ED INC,1,0
Bank Amor Corp Com Solicited Order Discretion Exercised,1,0


#### 1.2.3) Accounting for naming discrepancies in (a)

The method above is not bulletproof due to naming discrepancies, so we must account for this. We go over the remaining missing-ticker list manually in `TickerReview.xlsx`, identify the equities in the list and find their corresponding ticker. The tickers are entered in the sheet `2026ManuallyReviewed`, which we read back into the dataframe here.

Reasoning: we have observed some instances where there are missing tickers in the dataset despite being equities.

In [10]:
manual_tickers_2026 = pd.read_excel("TickerReview.xlsx", sheet_name="2026ManuallyReviewed")
manual_lookup_2026 = manual_tickers_2026.set_index("Company")["Ticker"]

# Scoped to exclude non-equity rows, so a ticker entered there for a fund/bond can't undo the clearing above.
target_2026 = trump2026["Ticker"].isna() & ~trump2026["IsNonEquity"]
trump2026.loc[target_2026, "Ticker"] = trump2026.loc[target_2026, "Ticker"].fillna(
    trump2026.loc[target_2026, "Company"].map(manual_lookup_2026)
)

→ We now have a complete list of the tickers of equity investments in 2026.

## 2) Adding tickers to the 2025 dataset

The 2025 dataset has no ticker symbols at all. We add them in a sequence of passes, from the most to the least reliable:

* We download the equity list from NYSE and NASDAQ (`TickersDB.xlsx`) and standardize the company names across 2025, 2026 and TickersDB (2.1).
* a) Exact match: 2025 company name → 2026 ticker (2.2)
* b) Excluding non-equity holdings from 2025 (2.3)
* c) Sequential prefix matching for whatever is left: INC → CORP → PLC (2.4)
* d) Suggesting candidates for the remainder via character-similarity match against TickersDB (2.5-2.6)
* e) Correcting and manual review of the suggested candidates (2.7)

Each step only touches the rows the previous steps left open. Whenever a match is ambiguous - a name or prefix that maps to more than one ticker, or where the two sources disagree - we exclude it rather than guess.

### 2.1) Standardizing company names

We load the NYSE/NASDAQ equity list `TickersDB.xlsx` and standardize the company names across 2025, 2026 and TickersDB ("Corporation" → "CORP", "Incorporated" → "INC", "Public Limited Company" → "PLC"), so the same company spelled differently in each source still matches.

In [11]:
import re

tickers_db = pd.read_excel("RawData/TickersDB.xlsx", sheet_name="NASDAQ_NYSE")
tickers_db = tickers_db.rename(columns={"Name": "Company"})[["Ticker", "Company"]].dropna()
print(f"TickersDB loaded: {len(tickers_db)} rows")

def standardize_company(name: str) -> str:
    s = str(name).upper().strip()
    s = s.replace(".", "").replace(",", "")
    s = re.sub(r"\bINCORPORATED\b", "INC", s)
    s = re.sub(r"\bCORPORATION\b", "CORP", s)
    s = re.sub(r"\bPUBLIC LIMITED COMPANY\b", "PLC", s)
    s = re.sub(r"\s+", " ", s).strip()
    return s

trump2025["CompanyStd"] = trump2025["Company"].apply(standardize_company)
trump2026["CompanyStd"] = trump2026["Company"].apply(standardize_company)
tickers_db["CompanyStd"] = tickers_db["Company"].apply(standardize_company)

print(trump2025[["Company", "CompanyStd"]].head())

TickersDB loaded: 7187 rows


                    Company                CompanyStd
0       ELEVANCE HEALTH INC       ELEVANCE HEALTH INC
1  CONSTELLATION BRANDS INC  CONSTELLATION BRANDS INC
2                BIOGEN INC                BIOGEN INC
3     UNITED PARCEL SVC INC     UNITED PARCEL SVC INC
4     UBER TECHNOLOGIES INC     UBER TECHNOLOGIES INC


### 2.2) Exact match: 2025 company name → 2026 ticker

If an investment was made in 2026 with the same (standardized) company name, we assume the same ticker and carry it over. Non-equity rows flagged in 1.2.2 are excluded, and a company name that maps to more than one ticker in 2026 is excluded rather than guessed.

Reasoning: this is the most reliable source, since it is literally the same disclosure process, same company and same ticker convention.

In [12]:
trump2025["Ticker"] = pd.Series(pd.NA, index=trump2025.index, dtype="object")

def build_exact_lookup(df):
    """Full CompanyStd -> ticker lookup, excluding names that map to >1 distinct ticker."""
    tmp = df.dropna(subset=["Ticker"])
    counts = tmp.groupby("CompanyStd")["Ticker"].nunique()
    colliding = set(counts[counts > 1].index)
    lookup = tmp[~tmp["CompanyStd"].isin(colliding)].groupby("CompanyStd")["Ticker"].first()
    return lookup, colliding

exact_lookup_2026, exact_collisions_2026 = build_exact_lookup(trump2026)

target_2025 = ~trump2025["IsNonEquity"]
before = trump2025["Ticker"].notna().sum()
trump2025.loc[target_2025, "Ticker"] = trump2025.loc[target_2025, "Ticker"].fillna(
    trump2025.loc[target_2025, "CompanyStd"].map(exact_lookup_2026)
)
after = trump2025["Ticker"].notna().sum()

print(f"Exact match against 2026: {after - before} rows matched; "
      f"{len(exact_collisions_2026)} colliding company names excluded")

Exact match against 2026: 14971 rows matched; 4 colliding company names excluded


### 2.3) Excluding non-equity holdings from 2025

Bonds, municipal notes and money-market/index funds have no stock ticker to find, so they are flagged (regex on patterns like "DUE", "MUNI", bond series codes, coupon %, maturity dates) and excluded from every matching step below. The flag itself was computed in 1.2.2; here we define `eligible_2025` from it.

In [13]:
eligible_2025 = ~trump2025["IsNonEquity"]

### 2.4) Sequential prefix matching for whatever is left: INC → CORP → PLC

For the companies the exact match missed, we cut the name at the first occurrence of a legal-entity token (INC, then CORP, then PLC) and exact-match the text before the token against both the 2026 dataset and TickersDB. Each pass only looks at rows still open after the previous one. Only exact matches are mapped to the corresponding ticker - names that map to more than one ticker (within a source, or disagreeing between the two sources) are excluded.

In [14]:
def cut_before(name, token):
    m = re.search(rf"\b{token}\b", name)
    return name[:m.start()].strip().rstrip(",.").strip() if m else None

def build_prefix_lookup(df, token):
    """Prefix -> ticker lookup from one source, excluding prefixes that map to >1 distinct ticker."""
    tmp = df.dropna(subset=["Ticker"]).copy()
    tmp["Prefix"] = tmp["CompanyStd"].apply(lambda s: cut_before(s, token))
    tmp = tmp.dropna(subset=["Prefix"])
    counts = tmp.groupby("Prefix")["Ticker"].nunique()
    colliding = set(counts[counts > 1].index)
    lookup = tmp[~tmp["Prefix"].isin(colliding)].groupby("Prefix")["Ticker"].first()
    return lookup, colliding

def sequential_match(token, still_open_mask):
    lookup_2026, collide_2026 = build_prefix_lookup(trump2026, token)
    lookup_tickersdb, collide_tickersdb = build_prefix_lookup(tickers_db, token)

    # Combine: must agree if both sources have an answer; disagreement -> ambiguous/excluded.
    combined = pd.DataFrame({"from_2026": lookup_2026, "from_tickersdb": lookup_tickersdb})
    agree_mask = (
        combined["from_2026"].isna() | combined["from_tickersdb"].isna()
        | (combined["from_2026"] == combined["from_tickersdb"])
    )
    combined = combined[agree_mask]
    combined["Ticker"] = combined["from_2026"].fillna(combined["from_tickersdb"])
    final_lookup = combined["Ticker"]

    disagreements = (~agree_mask).sum()
    all_collisions = collide_2026 | collide_tickersdb

    target_mask = still_open_mask & trump2025["CompanyStd"].str.contains(rf"\b{token}\b", regex=True, na=False)
    trump2025["Prefix_tmp"] = trump2025["CompanyStd"].apply(lambda s: cut_before(s, token))

    before = trump2025["Ticker"].notna().sum()
    trump2025.loc[target_mask, "Ticker"] = trump2025.loc[target_mask, "Ticker"].fillna(
        trump2025.loc[target_mask, "Prefix_tmp"].map(final_lookup)
    )
    after = trump2025["Ticker"].notna().sum()
    trump2025.drop(columns=["Prefix_tmp"], inplace=True)

    print(f"[{token}] candidates considered: {target_mask.sum()} rows; "
          f"newly matched: {after - before}; "
          f"collisions excluded: {len(all_collisions)}; cross-source disagreements excluded: {disagreements}")
    return trump2025["Ticker"].isna() & eligible_2025

In [15]:
still_open_2025 = trump2025["Ticker"].isna() & eligible_2025
still_open_2025 = sequential_match("INC", still_open_2025)
still_open_2025 = sequential_match("CORP", still_open_2025)
still_open_2025 = sequential_match("PLC", still_open_2025)

matched_2025 = trump2025["Ticker"].notna().sum()
equity_rows_2025 = eligible_2025.sum()
matched_of_equity_2025 = (trump2025["Ticker"].notna() & eligible_2025).sum()
print(f"\nTotal matched: {matched_2025} of {len(trump2025)} rows")
print(f"Matched of equity rows: {matched_of_equity_2025} of {equity_rows_2025}")
print(f"Still open & eligible: {still_open_2025.sum()} rows, "
      f"{trump2025.loc[still_open_2025, 'Company'].nunique()} distinct companies")

[INC] candidates considered: 3039 rows; newly matched: 2253; collisions excluded: 273; cross-source disagreements excluded: 6


[CORP] candidates considered: 1273 rows; newly matched: 860; collisions excluded: 261; cross-source disagreements excluded: 2


[PLC] candidates considered: 150 rows; newly matched: 132; collisions excluded: 9; cross-source disagreements excluded: 0

Total matched: 18216 of 21285 rows
Matched of equity rows: 18216 of 21102
Still open & eligible: 2886 rows, 573 distinct companies


### 2.5) Collecting the remainder

Companies without an INC/CORP/PLC token, or with an ambiguous match, are collected here and carried into the next step.

In [16]:
remaining_2025 = trump2025.loc[still_open_2025, ["Company", "CompanyStd"]].drop_duplicates().sort_values("Company")
print(f"{len(remaining_2025)} distinct companies still unmatched -- carrying these into the next step.")

573 distinct companies still unmatched -- carrying these into the next step.


### 2.6) Suggesting candidates for the remainder via character-similarity match against TickersDB

For whatever is still open after 2.4, we cut at a broader set of suffixes (INC, CORP, PLC, CO, LTD, TR, GROUP, HLDGS) and find the best letter-by-letter similarity match (`rapidfuzz`) against TickersDB, scored 0-100. This catches near-misses the exact match could not, e.g. `ABERCROMBIE & FITCH CO` vs. `ABERCROMBIE AND FITCH CO`. The suggestion is never applied automatically; we export it to `TickerReview.xlsx` together with the score and the matched TickersDB name, and review it in the next step.

In [17]:
from rapidfuzz import fuzz, process

BROAD_SUFFIX_TOKENS = ["INC", "CORP", "PLC", "CO", "LTD", "TR", "GROUP", "HLDGS"]

def cut_before_any(name, tokens=BROAD_SUFFIX_TOKENS):
    """Like cut_before, but cuts at whichever of several tokens occurs first; falls back to the full name."""
    best_idx = None
    for token in tokens:
        m = re.search(rf"\b{token}\b", name)
        if m and (best_idx is None or m.start() < best_idx):
            best_idx = m.start()
    return name[:best_idx].strip().rstrip(",.").strip() if best_idx is not None else name

# Candidate pool: TickersDB only -- 2026 has already been fully exploited by the exact-match passes above.
tickers_db["Prefix"] = tickers_db["CompanyStd"].apply(cut_before_any)
candidate_lookup_tickersdb = tickers_db.dropna(subset=["Ticker"]).groupby("Prefix")["Ticker"].first()
candidates = list(candidate_lookup_tickersdb.index)

def best_fuzzy_match(prefix):
    if not candidates or not prefix:
        return None
    return process.extractOne(prefix, candidates, scorer=fuzz.ratio)

remaining_2025["Prefix"] = remaining_2025["CompanyStd"].apply(cut_before_any)
fuzzy_results = remaining_2025["Prefix"].apply(best_fuzzy_match)
remaining_2025["SuggestedCompany"] = fuzzy_results.apply(lambda m: m[0] if m else None)
remaining_2025["Accuracy"] = fuzzy_results.apply(lambda m: round(m[1], 1) if m else None)
remaining_2025["SuggestedTicker"] = remaining_2025["SuggestedCompany"].map(candidate_lookup_tickersdb)

remaining_2025 = remaining_2025.sort_values("Accuracy", ascending=False)[
    ["Company", "Prefix", "SuggestedCompany", "SuggestedTicker", "Accuracy"]
]

with pd.ExcelWriter("TickerReview.xlsx", engine="openpyxl", mode="a", if_sheet_exists="replace") as writer:
    remaining_2025.to_excel(writer, sheet_name="2025RemainingManual", index=False)

print(f"{len(remaining_2025)} distinct companies written to the '2025RemainingManual' sheet, "
      f"each with a suggested ticker and accuracy score for manual confirmation.")
display(remaining_2025.head(20))

573 distinct companies written to the '2025RemainingManual' sheet, each with a suggested ticker and accuracy score for manual confirmation.


,Company,Prefix,SuggestedCompany,SuggestedTicker,Accuracy
1063,XEROX HOLDINGS CORP,XEROX HOLDINGS,XEROX HOLDINGS,XRX,100.0
5702,WYNDHAM HOTELS & RESORTS,WYNDHAM HOTELS & RESORTS,WYNDHAM HOTELS & RESORTS,WH,100.0
5817,ABERCROMBIE & FITCH CO,ABERCROMBIE & FITCH,ABERCROMBIE & FITCH,ANF,100.0
11294,WR BERKLEY CORP,WR BERKLEY,WR BERKLEY,WRB^E,100.0
2403,WATTS WATER TECHNOLOGIES,WATTS WATER TECHNOLOGIES,WATTS WATER TECHNOLOGIES,WTS,100.0
3464,UNUM GROUP,UNUM,UNUM,UNM,100.0
8300,US BANCORP,US BANCORP,US BANCORP,USB,100.0
14988,AMERICAN EXPRESS COMPANY,AMERICAN EXPRESS COMPANY,AMERICAN EXPRESS COMPANY,AXP,100.0
16067,AMERICAN HOMES 4 RENT,AMERICAN HOMES 4 RENT,AMERICAN HOMES 4 RENT,AMH,100.0
590,AMERIS BANCORP,AMERIS BANCORP,AMERIS BANCORP,ABCB,100.0


### 2.7) Correcting and manual review of the suggested candidates

We went over the 575 suggested rows manually in the sheet `2025ManuallyReviewed`. Some of the suggestions were correct, others were wrong and corrected, and some were non-equities. Here we fill in the reviewed `TickerFinal` for each company still missing a ticker; companies without a real match (blank `TickerFinal`, e.g. ETFs or non-equities like `WORLD GOLD TRUST SPDR GOLD MINISHARES`) are skipped. What is left without a ticker after this step is almost entirely genuine non-equities.

→ We now have a complete list of investments throughout 2025 with ticker names.

In [18]:
reviewed_2025 = pd.read_excel("TickerReview.xlsx", sheet_name="2025ManuallyReviewed")
ticker_final_lookup_2025 = reviewed_2025.set_index("Company")["TickerFinal"]

# Scoped to eligible_2025 so a stale review-sheet entry can't hand a ticker to a company
# that's since been reclassified as non-equity (e.g. after a NON_EQUITY_PATTERN update).
target_2025 = trump2025["Ticker"].isna() & eligible_2025
before = trump2025["Ticker"].notna().sum()
trump2025.loc[target_2025, "Ticker"] = trump2025.loc[target_2025, "Ticker"].fillna(
    trump2025.loc[target_2025, "Company"].map(ticker_final_lookup_2025)
)
after = trump2025["Ticker"].notna().sum()
print(f"Manually reviewed remainder: {after - before} rows filled in")

matched_2025 = trump2025["Ticker"].notna().sum()
equity_rows_2025 = eligible_2025.sum()
matched_of_equity_2025 = (trump2025["Ticker"].notna() & eligible_2025).sum()
print(f"\nTotal matched: {matched_2025} of {len(trump2025)} rows")
print(f"Matched of equity rows: {matched_of_equity_2025} of {equity_rows_2025}")

still_missing_2025 = trump2025.loc[eligible_2025 & trump2025["Ticker"].isna(), "Company"].drop_duplicates()
print(f"Equity rows still missing a ticker: {len(still_missing_2025)} distinct companies")
if len(still_missing_2025):
    display(still_missing_2025)

Manually reviewed remainder: 2881 rows filled in

Total matched: 21097 of 21285 rows
Matched of equity rows: 21097 of 21102
Equity rows still missing a ticker: 1 distinct companies


14775    WORLD GOLD TRUST SPDR GOLD MINISHARES
Name: Company, dtype: str

## 3) Merging the datasets

Both datasets now have tickers and the same column names and types. We union them into one dataframe, `Trump_trades_2025_2026`, and keep the trades from 20.01.2025 (inauguration day) onwards. Rows still missing a ticker are dropped, as they are non-equity holdings (ETFs, bonds, money-market funds, etc.).

In [19]:
# merging dataset
trump2025_final = trump2025[["Company", "Ticker", "Transaction", "Traded", "amount_low", "amount_high"]].copy()
trump2026_final = trump2026[["Company", "Ticker", "Transaction", "Traded", "amount_low", "amount_high"]].copy()
Trump_trades_2025_2026 = pd.concat([trump2025_final, trump2026_final], ignore_index=True)

# statistics on the combined dataset
print(f"Combined dataset: {len(Trump_trades_2025_2026)} rows "
      f"({len(trump2025_final)} from 2025, {len(trump2026_final)} from 2026)")
print(f"Rows still missing a ticker: {Trump_trades_2025_2026['Ticker'].isna().sum()} "
      f"(ETFs, indices, bonds, and other non-equity holdings)")

# Limiting to presidential trades only (after 20.01.2025)
cutoff = pd.Timestamp("2025-01-20")
before_rows = len(Trump_trades_2025_2026)
Trump_trades_2025_2026 = Trump_trades_2025_2026[Trump_trades_2025_2026["Traded"] >= cutoff].reset_index(drop=True)
print(f"Dropped rows traded before 20.01.2025:  {before_rows - len(Trump_trades_2025_2026)}")
# Limiting to equities only (non-equity holdings have no ticker to find)
print(f"Non-equity holdings dropped: {Trump_trades_2025_2026['Ticker'].isna().sum()}")
Trump_trades_2025_2026 = Trump_trades_2025_2026.dropna(subset=["Ticker"])

Trump_trades_2025_2026.to_excel("Trump_trades_2025_2026.xlsx", index=False)
print(f"Equities investments after inauguration: {len(Trump_trades_2025_2026)}")

Combined dataset: 28282 rows (21285 from 2025, 6997 from 2026)
Rows still missing a ticker: 529 (ETFs, indices, bonds, and other non-equity holdings)
Dropped rows traded before 20.01.2025:  83
Non-equity holdings dropped: 528


Equities investments after inauguration: 27671


## 4) Handling and standardizing different ticker symbols for a given corporation

Given the procedure above, we have instances where some companies have multiple tickers, for instance BRK.A and BRK.B due to the regex matching procedure, or GOOG and GOOGL. We standardize these tickers such that one company does not have multiple tickers, using the commonly used or representative equity ticker.

1) Looking at tickers with special characters (4.1-4.2)
2) Retrieving the corporation name from FMP from the ticker (4.3-4.4)
3) Groupby company name and analyzing different ticker symbols (4.5)
4) Analysis and correction of companies with multiple tickers (4.6)

### 4.1) Looking at tickers with special characters

Manual analysis of the stocks containing special characters, to ensure they are standardized to the same ticker. We standardize the ticker casing and list every unique ticker in `Trump_trades_2025_2026` that contains a special character (`.`, `#`, `^`, `/`, ...) together with the company names it appears under. Nothing is changed yet.

In [20]:
# Standardize ticker case first, then inspect punctuation in the main dataframe.
Trump_trades_2025_2026["Ticker"] = (
    Trump_trades_2025_2026["Ticker"].astype("string").str.strip().str.upper()
)
special_rows = Trump_trades_2025_2026[
    Trump_trades_2025_2026["Ticker"].str.contains(
        r"[^A-Z0-9]", regex=True, na=False
    )
].copy()

special_tickers = (
    special_rows.rename(columns={"Ticker": "OriginalTicker"})
    .groupby("OriginalTicker", as_index=False)
    .agg(
        SourceCompanyNames=(
            "Company",
            lambda values: " | ".join(sorted(set(values.dropna().astype(str)))),
        )
    )
    .sort_values("OriginalTicker")
    .reset_index(drop=True)
)

display(special_tickers)

,OriginalTicker,SourceCompanyNames
0,BF.A,BROWN-FORMAN CORP CL A | BROWN-FORMAN CORP CL ...
1,BF.B,BROWN FORMAN CORP | BROWN FORMAN CORP CLASS B
2,BRK.B,BERKSHIRE HATHAWAY | BERKSHIRE HATHAWAY CLASS ...
3,JPM^C,J P MORGAN CHASE AND CO
4,MOG.A,MOOG INC | MOOG INC CLASS A
5,SOLV#,SOLVENTUM CORP | SOLVENTUM CORPORATION
6,WRB^E,WR BERKLEY CORP


### 4.2) Applying the reviewed ticker changes

We map the special-character tickers manually to the FMP ticker convention (share classes with a hyphen, e.g. BRK.B → BRK-B) or to the representative equity ticker (e.g. the preferred share JPM^C → JPM), and overwrite them directly in `Trump_trades_2025_2026`.

In [21]:
reviewed_ticker_map = {
    "BF.A": "BF-B",
    "BF.B": "BF-B",
    "BRK.B": "BRK-B",
    "JPM^C": "JPM",
    "MOG.A": "MOG-A",
    "SOLV#": "SOLV",
    "WRB^E": "WRB",
}

unmapped_tickers = sorted(
    set(special_tickers["OriginalTicker"]) - set(reviewed_ticker_map)
)
if unmapped_tickers:
    raise ValueError(f"Special tickers missing from mapping: {unmapped_tickers}")

special_tickers["ReviewedTicker"] = special_tickers["OriginalTicker"].map(
    reviewed_ticker_map
)
Trump_trades_2025_2026["Ticker"] = Trump_trades_2025_2026["Ticker"].replace(
    reviewed_ticker_map
)

display(special_tickers)

,OriginalTicker,SourceCompanyNames,ReviewedTicker
0,BF.A,BROWN-FORMAN CORP CL A | BROWN-FORMAN CORP CL ...,BF-B
1,BF.B,BROWN FORMAN CORP | BROWN FORMAN CORP CLASS B,BF-B
2,BRK.B,BERKSHIRE HATHAWAY | BERKSHIRE HATHAWAY CLASS ...,BRK-B
3,JPM^C,J P MORGAN CHASE AND CO,JPM
4,MOG.A,MOOG INC | MOOG INC CLASS A,MOG-A
5,SOLV#,SOLVENTUM CORP | SOLVENTUM CORPORATION,SOLV
6,WRB^E,WR BERKLEY CORP,WRB


### 4.3) Retrieving the corporation name from FMP from the ticker

We look up every unique ticker in `Trump_trades_2025_2026` at FMP once and export the result to `FMP_company_names.xlsx`. If that file already exists, the retrieval is skipped and we read the names from the file instead (delete the file to fetch again). The result is a new column `FMPCompanyName`; tickers FMP does not know get `NA` and are listed for the manual correction in 4.4.

In [22]:
import json
import time
from pathlib import Path
from urllib.error import HTTPError, URLError
from urllib.parse import urlencode
from urllib.request import urlopen

FMP_SEARCH_URL = "https://financialmodelingprep.com/stable/search-symbol"
FMP_LOOKUP_FILE = Path("RawData/FMP_company_names.xlsx")  # all unique tickers + FMP names; delete it to fetch again

fmp_errors = set()  # HTTP errors seen during the lookup, reported at the end instead of being hidden


def lookup_company_name(ticker):
    """Company name from FMP for an exact ticker match, otherwise 'NA'."""
    url = f"{FMP_SEARCH_URL}?{urlencode({'query': ticker, 'apikey': fmp_api_key})}"
    for attempt in range(3):
        try:
            with urlopen(url, timeout=20) as response:
                results = json.loads(response.read().decode("utf-8"))
            for result in results:
                if str(result.get("symbol", "")).upper() == ticker:
                    return result.get("name") or "NA"
            return "NA"  # FMP answered, but does not know this ticker
        except HTTPError as error:
            if error.code != 429 and error.code < 500:  # e.g. 401/402/403 -> retrying will not help
                fmp_errors.add(f"HTTP {error.code}")
                return "NA"
        except (URLError, TimeoutError, OSError):
            pass
        time.sleep(attempt + 1)  # retry after 429 / 5xx / network problems
    fmp_errors.add("request failed after 3 attempts")
    return "NA"


# API key from .env (notebook folder or project root); never printed. Also used by 4.4.
fmp_api_key = None
for env_path in (Path.cwd() / ".env", Path.cwd().parent / ".env"):
    if env_path.is_file():
        for line in env_path.read_text(encoding="utf-8").splitlines():
            key, separator, value = line.partition("=")
            if separator and key.strip() == "FMP_API_KEY":
                fmp_api_key = value.strip().strip("\"'")
    if fmp_api_key:
        break
if not fmp_api_key:
    raise RuntimeError("FMP_API_KEY was not found in .env.")

unique_tickers = sorted(Trump_trades_2025_2026["Ticker"].dropna().unique())

if FMP_LOOKUP_FILE.is_file():
    # Already retrieved earlier -> skip FMP and reuse the file
    fmp_lookup = pd.read_excel(FMP_LOOKUP_FILE, keep_default_na=False)
    print(f"{FMP_LOOKUP_FILE} exists with {len(fmp_lookup)} tickers - FMP retrieval skipped")
else:
    # One FMP call per unique ticker in the main dataframe
    print(f"Looking up {len(unique_tickers)} unique tickers at FMP (roughly {len(unique_tickers) * 0.5 / 60:.0f} min)")
    fmp_company_names = []
    for index, ticker in enumerate(unique_tickers, start=1):
        fmp_company_names.append(lookup_company_name(ticker))
        if index % 50 == 0 or index == len(unique_tickers):
            print(f"[{index}/{len(unique_tickers)}]")
        time.sleep(0.25)
    if fmp_errors:
        print("WARNING - FMP returned errors; affected tickers are marked NA:", ", ".join(sorted(fmp_errors)))

    # Export all tickers with their FMP names, so the next run can skip the retrieval
    fmp_lookup = pd.DataFrame({"Ticker": unique_tickers, "FMPCompanyName": fmp_company_names})
    fmp_lookup.to_excel(FMP_LOOKUP_FILE, index=False)
    print(f"Saved {FMP_LOOKUP_FILE}")

# New column in the main dataframe: FMP company name, or "NA" when FMP did not know the ticker
Trump_trades_2025_2026["FMPCompanyName"] = (
    Trump_trades_2025_2026["Ticker"].map(fmp_lookup.set_index("Ticker")["FMPCompanyName"]).fillna("NA")
)

# Tickers to overwrite manually in 4.4
na_tickers = (
    Trump_trades_2025_2026[Trump_trades_2025_2026["FMPCompanyName"] == "NA"]
    .groupby("Ticker")["Company"]
    .agg(lambda names: " | ".join(sorted(set(names.dropna().astype(str)))))
    .to_frame("SourceCompanyNames")
)
print(f"Tickers not found at FMP: {len(na_tickers)} of {len(unique_tickers)}")
display(na_tickers)

RawData\FMP_company_names.xlsx exists with 1677 tickers - FMP retrieval skipped
Tickers not found at FMP: 8 of 1639


,SourceCompanyNames
Ticker,
AMB,ABM INDS INC
HWFA,HERITAGE FINL CORP WASH
MDEP,MEDPACE HLDGS INC
QW,QNITY ELECTRONICS INC | QNITY ELECTRONICS INC ...
RALW,RALLIANT CORP
SPA,PUBLIC STORAGE
VGNTW,VERSIGENT LTD F
WTC,STEWART INFORMATION SVCS


### 4.4) Manual analysis and ticker correction of those not found

The tickers FMP did not know are typos or when-issued suffixes in the source data (compare with `SourceCompanyNames` above). We overwrite them directly in `Trump_trades_2025_2026`. Corrected tickers that are not yet in `FMP_company_names.xlsx` are looked up once and appended to the file, after which `FMPCompanyName` is refreshed.

In [23]:
corrected_ticker_map = {
    "AMB": "ABM",
    "HWFA": "HFWA",
    "MDEP": "MEDP",
    "QW": "Q",
    "RALW": "RAL",
    "SPA": "PSA",
    "VGNTW": "VGNT",
    "WTC": "STC",
}

# Overwrite the tickers in the main dataframe
Trump_trades_2025_2026["Ticker"] = Trump_trades_2025_2026["Ticker"].replace(corrected_ticker_map)

# Corrected tickers that are not in FMP_company_names.xlsx yet -> look them up once and append to the file
new_tickers = sorted(set(corrected_ticker_map.values()) - set(fmp_lookup["Ticker"]))
if new_tickers:
    print(f"Looking up {len(new_tickers)} corrected tickers at FMP: {new_tickers}")
    new_rows = pd.DataFrame({"Ticker": new_tickers, "FMPCompanyName": [lookup_company_name(t) for t in new_tickers]})
    fmp_lookup = pd.concat([fmp_lookup, new_rows], ignore_index=True).sort_values("Ticker").reset_index(drop=True)
    fmp_lookup.to_excel(FMP_LOOKUP_FILE, index=False)

# Refresh the FMP company names after the ticker correction
Trump_trades_2025_2026["FMPCompanyName"] = (
    Trump_trades_2025_2026["Ticker"].map(fmp_lookup.set_index("Ticker")["FMPCompanyName"]).fillna("NA")
)

still_na = sorted(Trump_trades_2025_2026.loc[Trump_trades_2025_2026["FMPCompanyName"] == "NA", "Ticker"].unique())
print(f"Tickers still NA after correction: {still_na or 'none'}")
display(
    Trump_trades_2025_2026[Trump_trades_2025_2026["Ticker"].isin(corrected_ticker_map.values())][["Ticker", "FMPCompanyName"]]
    .drop_duplicates()
    .sort_values("Ticker")
)

Tickers still NA after correction: none


,Ticker,FMPCompanyName
567,ABM,ABM Industries Inc.
857,HFWA,Heritage Financial Corporation
5674,MEDP,"Medpace Holdings, Inc."
204,PSA,Public Storage
18470,Q,"Qnity Electronics, Inc."
11719,RAL,Ralliant Corp
751,STC,Stewart Information Services Corporation
25818,VGNT,Versigent PLC


### 4.5) Groupby company name and analyzing different ticker symbols

We group the main dataframe by `FMPCompanyName` and list the different tickers each company appears under.

In [24]:
company_ticker_groups = (
    Trump_trades_2025_2026[
        Trump_trades_2025_2026["FMPCompanyName"].ne("NA")
    ]
    .groupby("FMPCompanyName", as_index=False)
    .agg(
        Tickers=("Ticker", lambda values: " | ".join(sorted(set(values)))),
        TickerCount=("Ticker", "nunique"),
        TradeRowCount=("Ticker", "size"),
    )
    .sort_values(["TickerCount", "FMPCompanyName"], ascending=[False, True])
    .reset_index(drop=True)
)

display(company_ticker_groups)

,FMPCompanyName,Tickers,TickerCount,TradeRowCount
0,"Marsh & McLennan Companies, Inc.",MMC | MRSH,2,47
1,Two Harbors Investment Corp.,TWO | TWOD,2,16
2,3M Company,MMM,1,27
3,A. O. Smith Corporation,AOS,1,1
4,"A10 Networks, Inc.",ATEN,1,12
...,...,...,...,...
1627,Zurn Elkay Water Solutions Corporation,ZWS,1,16
1628,"e.l.f. Beauty, Inc.",ELF,1,3
1629,eBay Inc.,EBAY,1,38
1630,ePlus inc.,PLUS,1,9


### 4.6) Analysis and correction of companies with multiple tickers

We filter to the companies with more than one ticker and go over them manually.

In [25]:
multi_ticker_companies = company_ticker_groups[
    company_ticker_groups["TickerCount"] > 1
].copy()

display(multi_ticker_companies)

,FMPCompanyName,Tickers,TickerCount,TradeRowCount
0,"Marsh & McLennan Companies, Inc.",MMC | MRSH,2,47
1,Two Harbors Investment Corp.,TWO | TWOD,2,16


- **Marsh & McLennan Companies, Inc. - MMC | MRSH:** ticker change during the period of analysis; the dates correspond to the ticker that was correct during that period, so we keep both.
- **Two Harbors Investment Corp. - TWO | TWOD:** TWOD refers to a bond, whereas TWO refers to the actual equity. The company name in Trump's filings indicates equity, not a bond, so we substitute TWOD with TWO.

In [26]:
# Replacing TWOD with TWO in the main dataframe as the Company Name indicates that it is not a bond - but equity.
Trump_trades_2025_2026["Ticker"] = Trump_trades_2025_2026["Ticker"].replace({"TWOD": "TWO"})

# Export the main dataframe (after all ticker corrections) and the review sheets
with pd.ExcelWriter(
    "Trump_trades_2025_2026.xlsx",
    engine="openpyxl",
    mode="a",
    if_sheet_exists="replace",
) as writer:
    Trump_trades_2025_2026.to_excel(writer, sheet_name="Sheet1", index=False)
    Trump_trades_2025_2026[["Ticker", "FMPCompanyName"]].drop_duplicates().sort_values("Ticker").to_excel(
        writer, sheet_name="TickerCompanyLookup", index=False
    )
    company_ticker_groups.to_excel(writer, sheet_name="CompanyTickerGroups", index=False)

### 4.7) Validating ticker assignments against the disclosed company name

Safeguard against a bad *source* ticker: QuiverQuant's 2026 feed occasionally assigns a ticker that does not match the disclosed company at all. For example, `MONDELEZ INTL INC` was assigned `MDLX`, whose `FMPCompanyName` (from 4.3) is `Medley LLC SR NT 26` -- an unrelated bond, not Mondelez. Because 2025 tickers are partly carried over from 2026 (2.2), such an error can leak into 2025 too.

This is a general failsafe, independent of how the ticker was assigned: for every unique (Company, Ticker) pair we fuzzy-match (`rapidfuzz.fuzz.token_set_ratio`) the standardized disclosed `Company` name against the standardized `FMPCompanyName` already on file, and flag low-similarity pairs for manual review. Most flagged rows turn out to be legitimate ticker renames (e.g. Raytheon -> RTX, Facebook -> Meta); a minority, like Mondelez, are genuine mismatches.

In [27]:
MISMATCH_SCORE_THRESHOLD = 60  # flag pairs scoring below this for manual review

unique_pairs = (
    Trump_trades_2025_2026[["Company", "Ticker", "FMPCompanyName"]]
    .drop_duplicates()
    .reset_index(drop=True)
)

def mismatch_score(company, fmp_name):
    if not fmp_name or fmp_name == "NA":
        return 0
    return fuzz.token_set_ratio(standardize_company(company), standardize_company(fmp_name))

unique_pairs["Score"] = unique_pairs.apply(
    lambda r: mismatch_score(r["Company"], r["FMPCompanyName"]), axis=1
)

ticker_company_mismatch = (
    unique_pairs[unique_pairs["Score"] < MISMATCH_SCORE_THRESHOLD]
    .sort_values("Score")
    .reset_index(drop=True)
)

with pd.ExcelWriter("TickerReview.xlsx", engine="openpyxl", mode="a", if_sheet_exists="replace") as writer:
    ticker_company_mismatch.to_excel(writer, sheet_name="TickerCompanyMismatch", index=False)

print(f"{len(unique_pairs)} unique (Company, Ticker) pairs checked; "
      f"{len(ticker_company_mismatch)} scored below {MISMATCH_SCORE_THRESHOLD} and were written to "
      "TickerReview.xlsx!TickerCompanyMismatch for manual review")
display(ticker_company_mismatch.head(20))

3448 unique (Company, Ticker) pairs checked; 78 scored below 60 and were written to TickerReview.xlsx!TickerCompanyMismatch for manual review


,Company,Ticker,FMPCompanyName,Score
0,General Elec Co Com Now Solicited Order Discre...,GE,GE Aerospace,15.384615
1,General Elec Co Com New Solicited Order Discre...,GE,GE Aerospace,18.348624
2,RAYTHEON TECHNOLOGIES,RTX,RTX Corporation,20.689655
3,WABTEC,WAB,Westinghouse Air Brake Technologies Corporation,21.739130
4,WABTEC INC,WAB,Westinghouse Air Brake Technologies Corporation,24.000000
5,FACEBOOK INC-A,META,"Meta Platforms, Inc.",25.000000
6,GROCERY OUTLET HLDG CORP,GOGO,Gogo Inc.,25.000000
7,LIBERTY MEDIA CORP DEL CLASS SERIES C,FWONA,Formula One Group,25.925926
8,FNF GROUP CLASS CLASS A,FNF,"Fidelity National Financial, Inc.",29.166667
9,FNF GROUP CLASS A,FNF,"Fidelity National Financial, Inc.",29.166667


### 4.8) Manual review and correction of flagged mismatches

We go over the `TickerCompanyMismatch` sheet by hand and save the decisions into a separate sheet, `TickerCompanyMismatchReviewed`, with three added columns: `Ticker ok` (`v` = the ticker is actually correct, FMP's name just looks different; `x` = genuinely wrong), `CorrectedTicker` (the real ticker, when one exists), and `Review note`. Two outcomes for an `x` row:

* **Has a `CorrectedTicker`** (e.g. Mondelez MDLX -> MDLZ, Broadridge BRK-B -> BR): every row with that exact `(Company, Ticker)` pair gets its `Ticker` replaced.
* **No `CorrectedTicker`** (e.g. `COF 3.95 PERP`, `NTRS 4.6`): there is no common-equity ticker to substitute because the security itself is not common equity (a perpetual preferred/note that slipped past the non-equity filter in 1.2.2). These rows are dropped instead of relabeled.

In [28]:
reviewed = pd.read_excel("TickerReview.xlsx", sheet_name="TickerCompanyMismatchReviewed")
flagged_wrong = reviewed[reviewed["Ticker ok"] == "x"]

has_fix = flagged_wrong["CorrectedTicker"].notna() & (flagged_wrong["CorrectedTicker"].astype(str).str.strip() != "")
corrections = flagged_wrong[has_fix]
no_equity_substitute = flagged_wrong[~has_fix]

# Substitute the ticker for every row sharing that exact (Company, old Ticker) pair
fixed_rows = 0
for _, review_row in corrections.iterrows():
    mask = (Trump_trades_2025_2026["Company"] == review_row["Company"]) & (
        Trump_trades_2025_2026["Ticker"] == review_row["Ticker"]
    )
    fixed_rows += mask.sum()
    Trump_trades_2025_2026.loc[mask, "Ticker"] = review_row["CorrectedTicker"]

# No valid equity ticker exists for these -- they are non-equity securities that slipped past 1.2.2; drop them
dropped_rows = 0
for _, review_row in no_equity_substitute.iterrows():
    mask = (Trump_trades_2025_2026["Company"] == review_row["Company"]) & (
        Trump_trades_2025_2026["Ticker"] == review_row["Ticker"]
    )
    dropped_rows += mask.sum()
    Trump_trades_2025_2026 = Trump_trades_2025_2026[~mask]
Trump_trades_2025_2026 = Trump_trades_2025_2026.reset_index(drop=True)

# Look up any corrected ticker not yet in FMP_company_names.xlsx, then refresh FMPCompanyName
new_corrected_tickers = sorted(set(corrections["CorrectedTicker"]) - set(fmp_lookup["Ticker"]))
if new_corrected_tickers:
    print(f"Looking up {len(new_corrected_tickers)} corrected tickers at FMP: {new_corrected_tickers}")
    new_rows = pd.DataFrame({
        "Ticker": new_corrected_tickers,
        "FMPCompanyName": [lookup_company_name(t) for t in new_corrected_tickers],
    })
    fmp_lookup = pd.concat([fmp_lookup, new_rows], ignore_index=True).sort_values("Ticker").reset_index(drop=True)
    fmp_lookup.to_excel(FMP_LOOKUP_FILE, index=False)

Trump_trades_2025_2026["FMPCompanyName"] = (
    Trump_trades_2025_2026["Ticker"].map(fmp_lookup.set_index("Ticker")["FMPCompanyName"]).fillna("NA")
)

print(f"Applied {len(corrections)} corrected ticker(s) across {fixed_rows} rows")
print(f"Dropped {len(no_equity_substitute)} non-equity pair(s) across {dropped_rows} rows (no common-equity substitute): {list(no_equity_substitute['Company'])}")

Applied 14 corrected ticker(s) across 116 rows
Dropped 2 non-equity pair(s) across 12 rows (no common-equity substitute): ['COF 3.95 PERP', 'NTRS 4.6']


### 4.9) Final non-equity sweep

The manual review in 4.8 surfaced two securities that are not common equity at all -- `COF 3.95 PERP` and `NTRS 4.6`, preferred/perpetual notes that slipped past the 1.2.2 regex because they use neither a `%` sign nor any of its existing keywords. We extend that regex with `PERP` and `PFD` (checked against the full dataset first -- these two tokens add only genuine non-equity preferred/perpetual securities, no false positives) and run it once more over the finished dataset as a last safeguard, before the final export.

In [29]:
NON_EQUITY_PATTERN_FINAL = re.compile(NON_EQUITY_PATTERN.pattern + r"|\bPERP\b|\bPFD\b", re.IGNORECASE)

final_non_equity_mask = Trump_trades_2025_2026["Company"].str.contains(
    NON_EQUITY_PATTERN_FINAL, regex=True, na=False
)
print(f"Final non-equity sweep: {final_non_equity_mask.sum()} rows caught, "
      f"{Trump_trades_2025_2026.loc[final_non_equity_mask, 'Company'].nunique()} distinct companies")
display(Trump_trades_2025_2026.loc[final_non_equity_mask, ["Company", "Ticker", "FMPCompanyName"]].drop_duplicates())

Trump_trades_2025_2026 = Trump_trades_2025_2026[~final_non_equity_mask].reset_index(drop=True)

# The state after sections 1-4 is kept and written by section 5 as the sheet TradesBeforePriceReview
trades_c7 = Trump_trades_2025_2026.copy()
print(f"Cleaned dataset after sections 1-4: {len(Trump_trades_2025_2026)} equity rows, {Trump_trades_2025_2026['Ticker'].nunique()} unique tickers")


Final non-equity sweep: 10 rows caught, 2 distinct companies


,Company,Ticker,FMPCompanyName
8198,BANK OF AMERICA CORP CONV PFD,BAC,Bank of America Corporation
8203,WELLS FARGO & CO PERP NT,WFC,Wells Fargo & Company


Cleaned dataset after sections 1-4: 27649 equity rows, 1629 unique tickers


→ We now have all the ticker names standardized for each company. Section 5 checks them against the price series.

## 5) Ticker corrections after price fetching

Fetching the price series for the tickers above (Step 3, folder `3) Prices`) showed three kinds of error that sections 1–4 could not see, because they only appear once a ticker is matched to a price series:

1. **Wrong company or stock.** A ticker whose price series belongs to another company than the one the disclosure names (`NCL` is Northann, the disclosure says Northern Trust; `HSAI` is Hesai, the disclosure says Hess). Six of these tickers are right for some rows and wrong for others (`BOX` is Box Inc. on 20 rows and Jack in the Box on 11), so the correction is made per (Ticker, Company) pair, never per ticker.
2. **Ticker changes.** Companies that changed their symbol inside or after the sample (`BK` became `BNY`, `MMC` became `MRSH`), and companies whose symbol was replaced by a successor's after a merger (`PARA` became `PSKY`, later `SKYD`). The price provider serves the whole history under the latest symbol, so every row of the old ticker moves to the last symbol in its change chain, whatever the trade date. One company, one ticker, which is what section 4 intended.
3. **Not shares, or not priceable.** A few disclosures under an equity ticker are notes of the company's financing arm (BlackRock Funding, BP Capital Markets); these rows are dropped. Five purchased companies have no usable price series at FMP under any symbol (their old symbol now belongs to an ETF, or the company re-registered after the purchases); their rows stay in the trade file but their purchases are left out of the purchase table in section 6.

Every decision was reviewed by hand (Andreas Borhaug and Filip, 2026-11-04/05) and is recorded in `TickerReview.xlsx`: sheet `WrongCompanyAfterPrices` holds the pairs, `NonEquity` the rows to drop, `Unpriceable` the five tickers, and `TickerRenames` is generated here from FMP's symbol-change history with a `Decision` column (`apply` / `check` / `skip`) in which a `skip` written by the reviewer survives a rerun. `RawData/FirstPriceFetch/` keeps the co-author's notes from the first fetch, the origin of this review. `RawData/FMP_symbol_changes.json` and `RawData/FMP_ticker_checks.json` cache FMP's answers (the symbol-change list; profile and first price date per symbol), so a rerun makes no call unless an entry is missing.

The logic flows as follows:

5.1 Validation of the review sheets · 5.2 Wrong company or stock, per pair · 5.3 Ticker changes, every row to the latest symbol · 5.4 Non-equity rows dropped · 5.5 FMP names for the new tickers · 5.6 Checks and export of `Trades` and `TradesBeforePriceReview`

### 5.1) Validation of the review sheets
Every (Ticker, Company) pair in the review sheets must exist in the cleaned trades exactly as written, every corrected ticker must be present, and no pair may be listed twice; otherwise the notebook stops here. FMP's symbol-change list (5,535 rows back to 1969, requested with `limit=10000`) is read from the cache, or fetched once and cached.

In [30]:
import json, re, time
from pathlib import Path
from urllib.error import HTTPError, URLError
from urllib.parse import urlencode
from urllib.request import urlopen

REVIEW_FILE = "TickerReview.xlsx"
SYMBOL_CHANGES_FILE = Path("RawData/FMP_symbol_changes.json")
TICKER_CHECKS_FILE = Path("RawData/FMP_ticker_checks.json")
OUT_FILE = "Trump_trades_2025_2026.xlsx"
PURCHASES_CSV = Path("../3) Prices/RawData/purchases.csv")      # the input of the price scripts
WINDOW_FROM, WINDOW_TO = "2025-01-20", "2026-08-31"   # sample window; symbol changes after it do not merge two companies

def fmp_get(endpoint, **params):
    url = f"https://financialmodelingprep.com/stable/{endpoint}?{urlencode({**params, 'apikey': fmp_api_key})}"
    for attempt in range(3):
        try:
            with urlopen(url, timeout=30) as resp:
                return json.loads(resp.read().decode("utf-8"))
        except (HTTPError, URLError, TimeoutError, OSError):
            time.sleep(attempt + 1)
    return None

trades = Trump_trades_2025_2026.copy()
trades["Traded"] = pd.to_datetime(trades["Traded"])
rows_before, tickers_before = len(trades), trades["Ticker"].nunique()
tickers_at_import = set(trades["Ticker"])
wrong = pd.read_excel(REVIEW_FILE, sheet_name="WrongCompanyAfterPrices")
non_equity = pd.read_excel(REVIEW_FILE, sheet_name="NonEquity")
unpriceable = pd.read_excel(REVIEW_FILE, sheet_name="Unpriceable")

if SYMBOL_CHANGES_FILE.exists():
    symbol_changes = json.load(open(SYMBOL_CHANGES_FILE, encoding="utf-8"))["rows"]
else:
    symbol_changes = fmp_get("symbol-change", limit=10000) or []
    json.dump({"endpoint": "stable/symbol-change?limit=10000", "fetched": str(pd.Timestamp.today().date()), "rows": symbol_changes}, open(SYMBOL_CHANGES_FILE, "w", encoding="utf-8"))
checks_cache = json.load(open(TICKER_CHECKS_FILE, encoding="utf-8")) if TICKER_CHECKS_FILE.exists() else {"note": "FMP profile and first price date per symbol", "symbols": {}}

print(f"Cleaned trades: {rows_before} rows, {tickers_before} tickers | WrongCompanyAfterPrices: {len(wrong)} pairs | NonEquity: {len(non_equity)} pairs | "
      f"Unpriceable: {len(unpriceable)} tickers | FMP symbol changes: {len(symbol_changes)} | cached symbol checks: {len(checks_cache['symbols'])}")

pairs_in_trades = set(zip(trades["Ticker"], trades["Company"]))
problems = []
for r in wrong.itertuples():
    if (r.Ticker, r.Company) not in pairs_in_trades:
        problems.append(f"WrongCompanyAfterPrices pair not in trades: {r.Ticker} / {r.Company}")
    if not isinstance(r.CorrectedTicker, str) or not r.CorrectedTicker.strip() or r.CorrectedTicker != r.CorrectedTicker.upper():
        problems.append(f"bad CorrectedTicker for {r.Ticker} / {r.Company}")
for r in non_equity.itertuples():
    if (r.Ticker, r.Company) not in pairs_in_trades:
        problems.append(f"NonEquity pair not in trades: {r.Ticker} / {r.Company}")
for sheet, df in [("WrongCompanyAfterPrices", wrong), ("NonEquity", non_equity)]:
    dups = df[df.duplicated(["Ticker", "Company"], keep=False)]
    if len(dups):
        problems.append(f"{sheet} pairs listed twice: {dups[['Ticker', 'Company']].values.tolist()}")
assert not problems, problems
print("review sheets valid")
display(wrong[["Ticker", "Company", "CorrectedTicker", "CorrectedCompanyName", "RowsInTrades"]])

Cleaned trades: 27649 rows, 1629 tickers | WrongCompanyAfterPrices: 27 pairs | NonEquity: 5 pairs | Unpriceable: 5 tickers | FMP symbol changes: 5535 | cached symbol checks: 1679
review sheets valid


,Ticker,Company,CorrectedTicker,CorrectedCompanyName,RowsInTrades
0,NCL,NORTHERN TR CORP,NTRS,Northern Trust Corporation,6
1,HSAI,HESS CORP,HES,Hess Corporation,6
2,MMED,INARI MED INC,NARI,"Inari Medical, Inc.",5
3,CNDT,CONFLUENT INC,CFLT,"Confluent, Inc.",2
4,AEVA,DYNAVAX TECHNOLOGIES COR,DVAX,Dynavax Technologies Corporation,11
5,BOX,JACK IN THE BOX INC,JACK,Jack in the Box Inc.,11
6,CBFV,PNC FINANCIAL SERVICES GROUP,PNC,"The PNC Financial Services Group, Inc.",6
7,CBFV,PNC FINANCIAL SERVICES GROUP I,PNC,"The PNC Financial Services Group, Inc.",2
8,COFS,DISCOVER FINANCIAL SERVICES,DFS,Discover Financial Services,2
9,HCA,ACADIA HEALTHCARE CO INC,ACHC,"Acadia Healthcare Company, Inc.",4


### 5.2) Wrong company or stock: per (Ticker, Company) pair
Each review row names the disclosed company exactly as it appears in the trades. Only rows with that exact (Ticker, Company) pair get the corrected ticker, as in 4.8; rows of the same ticker under the right company name keep it (Box Inc. stays `BOX`, HCA Healthcare stays `HCA`, Pool Corporation stays `POOL`, Rogers stays `ROG`, The Bancorp stays `TBBK`, First BanCorp Puerto Rico stays `FBP`).

In [31]:
trades["TickerBefore"] = trades["Ticker"]          # kept until the export, for the change log
changes_log = []  # (rule, old, new, company, rows)
for r in wrong.itertuples():
    mask = (trades["Ticker"] == r.Ticker) & (trades["Company"] == r.Company)
    trades.loc[mask, "Ticker"] = r.CorrectedTicker
    changes_log.append(("wrong_company", r.Ticker, r.CorrectedTicker, r.Company, int(mask.sum())))
print(f"Wrong company or stock: {len(wrong)} pairs, {sum(c[4] for c in changes_log)} rows re-tickered")
kept = trades[trades["Ticker"].isin(wrong["Ticker"].unique())].groupby(["Ticker", "Company"]).size()
print("\nRows that keep a ticker from the list (the right company under that ticker):")
display(kept.reset_index(name="rows"))

Wrong company or stock: 27 pairs, 184 rows re-tickered

Rows that keep a ticker from the list (the right company under that ticker):


,Ticker,Company,rows
0,BOX,BOX INC,16
1,BOX,BOX INC CLASS A,4
2,FBP,FIRST BANCORP P R F,16
3,HCA,HCA HEALTHCARE INC,19
4,HCA,"HCA HEALTHCARE, INC",2
5,HCA,"HCA HEALTHCARE, INC.",1
6,HCA,HCA HOLDINGS INC,4
7,POOL,POOL CORP,23
8,POOL,POOL CORPORATION,2
9,POOL,Pool Corporation,1


### 5.3) Ticker changes and merger successors: every row to the latest symbol

For every ticker now in the trades we follow FMP's symbol-change history forward in date order (changes since the start of the sample; rows for preferred shares, notes, depositary shares, warrants and when-issued symbols dropped) to the last symbol in the chain, and add the two when-issued pairs from the hand review (`HONAV` → `HONA`, `VSNTV` → `VSNT`); merger successors inside the universe are reached through the same chain (`PSKY` → `SKYD`). The result is the sheet `TickerRenames`, one row per old ticker, with the chain, the source and a `Decision`:

* pairs on the hand review list are `apply`;
* other chain rows are `apply` when the FMP profile of the latest symbol carries the same CIK as the old ticker (a plain rename, `EQR` → `VMRK`), or a different CIK while the change happened inside the sample window, the old ticker no longer trades and the new profile is an operating company, not a fund (a merger successor); otherwise `check` (skipped, listed for the reviewer): no FMP profile for the latest symbol (`AROC` → `IDLDX`), a symbol taken over by an ETF (`SPCX` → `SPCK`), or a successor whose takeover came after the sample, so that the two traded as separate companies throughout it (`INDV` → `SUPN`, 2026-11-03);
* **price coverage**: the latest symbol's FMP price history must start on or before the first trade of the old ticker; if it does not, the price provider has not carried the history over and the row is `check` (`HLX` → `HOS`: HOS prices only from 2026-08-04, trades from January 2025);
* a `skip` written in the sheet by the reviewer is kept on a rerun and vetoes the row permanently; `apply` and `check` are recomputed.

Every row of an `apply` ticker then gets the latest symbol, whatever its trade date. The profile and price-coverage checks are read from `RawData/FMP_ticker_checks.json` and fetched only for a symbol not yet cached.

In [32]:
HAND_LIST = {"HONAV": "HONA", "VSNTV": "VSNT", "CSWI": "CSW", "ASGN": "EFOR", "ATGE": "CVSA", "AXL": "DCH", "MMC": "MRSH", "BK": "BNY",
             "SJW": "HTO", "BHLB": "BBT", "NYMT": "ADAM", "AMBC": "OSG", "MODG": "CALY", "OMI": "ACH", "PSTG": "P", "IAC": "PPLI", "SCVL": "SHOE"}
JUNK = ("PREFERRED", "DEPOSITARY", "NOTES", "WARRANT", "UNITS", "DEBENTURE", "RIGHTS")

def when_issued(c):
    o, n = c.get("oldSymbol") or "", c.get("newSymbol") or ""
    return o.endswith(".V") or o == n + "V" or o.endswith("-WI")

usable = [c for c in symbol_changes if c["date"] >= WINDOW_FROM and not when_issued(c)
          and not any(j in (c.get("companyName") or "").upper() for j in JUNK)]
next_symbol = {}
for c in sorted(usable, key=lambda c: c["date"]):
    next_symbol.setdefault(c["oldSymbol"], []).append((c["date"], c["newSymbol"], c.get("companyName")))

def chain_from(ticker):
    hops, cur, last_date, seen = [], ticker, "", {ticker}
    while cur in next_symbol:
        later = [h for h in next_symbol[cur] if h[0] >= last_date and h[1] not in seen]
        if not later:
            break
        d, n, name = later[0]
        hops.append(f"{cur}->{n} ({d})"); seen.add(n); last_date, cur = d, n
    return cur, hops

def symbol_check(symbol):
    """FMP profile (cik, name, trading flag, fund flags) and first price date since 2024 for a symbol; cached in RawData/FMP_ticker_checks.json."""
    entry = checks_cache["symbols"].setdefault(symbol, {})
    if "profile" not in entry:
        data = fmp_get("profile", symbol=symbol)
        prof = data[0] if isinstance(data, list) and data else {}
        entry["profile"] = {k: prof.get(k) for k in ["symbol", "companyName", "cik", "isActivelyTrading", "isEtf", "isFund"]}
    if "first_price" not in entry:
        data = fmp_get("historical-price-eod/light", symbol=symbol, **{"from": "2024-01-01", "to": "2026-12-31"})
        dates = sorted(x["date"] for x in data) if isinstance(data, list) else []
        entry["first_price"] = dates[0] if dates else None
    return entry

current = sorted(trades["Ticker"].unique())
rename_rows = []
for t in current:
    latest, hops = chain_from(t)
    source = "FMP symbol-change"
    if latest == t and t in HAND_LIST:
        latest, hops, source = HAND_LIST[t], [f"{t}->{HAND_LIST[t]} (hand review)"], "hand review"
    if latest == t:
        continue
    if t in HAND_LIST:
        decision, check, kind = "apply", "hand review", "hand review: same company under the new symbol"
    else:
        prof = symbol_check(latest)["profile"]; old = symbol_check(t)["profile"]
        new_cik = str(prof.get("cik") or "").zfill(10) if prof.get("cik") else None
        old_cik = str(old.get("cik") or "").zfill(10) if old.get("cik") else None
        old_active = old.get("isActivelyTrading")
        is_fund = bool(re.search(r"(ETF|FUND|TRUST|ETN)", (prof.get("companyName") or "").upper())) or prof.get("isEtf") or prof.get("isFund")
        change_date = max((h.split("(")[-1].rstrip(")") for h in hops), default="")
        if not new_cik:
            decision, kind = "check", "no FMP profile for the latest symbol"
        elif new_cik == old_cik:
            decision, kind = "apply", "same CIK: rename"
        elif change_date > WINDOW_TO:
            decision, kind = "check", f"other CIK and the change ({change_date}) came after the sample: two separate companies throughout it"
        elif old_active is False and not is_fund:
            decision, kind = "apply", "other CIK, change inside the sample, old ticker no longer trading, new profile is a company: merger successor"
        else:
            decision, kind = "check", f"other CIK ({'fund' if is_fund else 'old ticker still trading'}): symbol taken over by another issuer?"
        check = f"CIK old {old_cik} / latest {new_cik} ({prof.get('companyName')}): {kind}"
    first_trade = str(trades.loc[trades["Ticker"] == t, "Traded"].min())[:10]
    first_price = symbol_check(latest)["first_price"]
    if decision == "apply" and (first_price is None or first_price > first_trade):
        decision = "check"
        check += f"; PRICE COVERAGE: {latest} prices start {first_price}, first trade {first_trade}"
        kind += f"; but {latest} prices start {first_price}, after the first trade {first_trade}"
    else:
        check += f"; prices from {first_price}"
    change_date = hops[-1].split("(")[-1].rstrip(")") if hops and re.search(r"\(\d{4}-\d\d-\d\d\)$", hops[-1]) else ""
    rename_rows.append(dict(OldTicker=t, LatestTicker=latest, Chain=" ; ".join(hops), Source=source, CikCheck=check,
                            RowsOld=int((trades["Ticker"] == t).sum()), Decision=decision, ChangeDate=change_date, Reason=kind))
renames = pd.DataFrame(rename_rows)
json.dump(checks_cache, open(TICKER_CHECKS_FILE, "w", encoding="utf-8"))

# a 'skip' and a ReviewNote written by the reviewer survive a rerun; apply / check and Reason are recomputed
renames["ReviewNote"] = ""
try:
    previous = pd.read_excel(REVIEW_FILE, sheet_name="TickerRenames").set_index("OldTicker")
    renames.loc[renames["OldTicker"].map(previous["Decision"]).eq("skip"), "Decision"] = "skip"
    if "ReviewNote" in previous:
        renames["ReviewNote"] = renames["OldTicker"].map(previous["ReviewNote"]).fillna("")
except ValueError:
    pass
with pd.ExcelWriter(REVIEW_FILE, engine="openpyxl", mode="a", if_sheet_exists="replace") as w:
    renames.to_excel(w, sheet_name="TickerRenames", index=False)
print(f"TickerRenames: {len(renames)} old tickers; decisions {renames['Decision'].value_counts().to_dict()}")
display(renames)

for r in renames[renames["Decision"] == "apply"].itertuples():
    mask = trades["Ticker"] == r.OldTicker
    trades.loc[mask, "Ticker"] = r.LatestTicker
    changes_log.append(("latest_ticker", r.OldTicker, r.LatestTicker, "(all rows)", int(mask.sum())))
print(f"\nLatest ticker: {int((renames['Decision'] == 'apply').sum())} tickers applied, {sum(c[4] for c in changes_log if c[0] == 'latest_ticker')} rows re-tickered; "
      f"skipped ({', '.join(renames.loc[renames['Decision'] != 'apply', 'OldTicker'])}) left as they are")

TickerRenames: 25 old tickers; decisions {'apply': 21, 'check': 2, 'skip': 2}


,OldTicker,LatestTicker,Chain,Source,CikCheck,RowsOld,Decision,ChangeDate,Reason,ReviewNote
0,AMBC,OSG,AMBC->OSG (2025-11-20),FMP symbol-change,hand review; prices from 2024-01-02,3,apply,2025-11-20,hand review: same company under the new symbol,Applied: 3 rows moved AMBC -> OSG. FMP serves ...
1,AROC,IDLDX,AROC->IDLDX (2025-05-27),FMP symbol-change,CIK old 0001389050 / latest None (None): no FM...,18,check,2025-05-27,no FMP profile for the latest symbol,"Not applied, rows stay AROC (18): FMP lists a ..."
2,ASGN,EFOR,ASGN->EFOR (2026-04-24),FMP symbol-change,hand review; prices from 2024-01-02,1,apply,2026-04-24,hand review: same company under the new symbol,Applied: 1 row moved ASGN -> EFOR. Price conti...
3,ATGE,CVSA,ATGE->CVSA (2026-02-24),FMP symbol-change,hand review; prices from 2024-01-02,10,apply,2026-02-24,hand review: same company under the new symbol,Applied: 10 rows moved ATGE -> CVSA. FMP serve...
4,AXL,DCH,AXL->DCH (2026-02-05),FMP symbol-change,hand review; prices from 2024-01-02,5,apply,2026-02-05,hand review: same company under the new symbol,Applied: 5 rows moved AXL -> DCH. FMP serves t...
5,BHLB,BBT,BHLB->BBT (2025-09-02),FMP symbol-change,hand review; prices from 2024-01-02,6,apply,2025-09-02,hand review: same company under the new symbol,Applied: 6 rows moved BHLB -> BBT. FMP serves ...
6,BK,BNY,BK->BNY (2026-05-21),FMP symbol-change,hand review; prices from 2024-01-02,23,apply,2026-05-21,hand review: same company under the new symbol,Applied: 23 rows moved BK -> BNY. FMP serves t...
7,CSWI,CSW,CSWI->CSW (2025-06-09),FMP symbol-change,hand review; prices from 2024-01-02,24,apply,2025-06-09,hand review: same company under the new symbol,Applied: 24 rows moved CSWI -> CSW. FMP serves...
8,EQR,VMRK,EQR->VMRK (2026-08-18),FMP symbol-change,CIK old 0000906107 / latest 0000906107 (Vivmar...,26,apply,2026-08-18,same CIK: rename,Applied: 26 rows moved EQR -> VMRK. FMP serves...
9,HLX,HOS,HLX->HOS (2026-09-02),FMP symbol-change,CIK old 0000866829 / latest 0000866829 (Hornbe...,12,skip,2026-09-02,same CIK: rename; but HOS prices start 2026-08...,"Skip (reviewer), rows stay HLX (12): Helix bec..."



Latest ticker: 21 tickers applied, 201 rows re-tickered; skipped (AROC, HLX, INDV, SPCX) left as they are


### 5.4) Non-equity rows: dropped
Disclosures that name a company's financing vehicle (BlackRock Funding Inc., BP Capital Markets plc) are purchases of notes, not of the parent's shares; the 4.9 sweep could not catch them because the names carry no bond vocabulary. The co-author's return check surfaced them, and the CEO decided which (Ticker, Company) pairs to drop; they are listed in sheet `NonEquity` with the reason and removed here, as 4.8 did for non-equity substitutes. The parent's own rows stay (BlackRock Inc. keeps `BLK`).

In [33]:
drop_mask = pd.Series(False, index=trades.index)
for r in non_equity.itertuples():
    m = (trades["Ticker"] == r.Ticker) & (trades["Company"] == r.Company)
    changes_log.append(("non_equity_dropped", r.Ticker, "(dropped)", r.Company, int(m.sum())))
    drop_mask |= m
dropped = trades[drop_mask].copy()
dropped_rows = int(drop_mask.sum())
trades = trades[~drop_mask].reset_index(drop=True)
print(f"Non-equity rows dropped: {dropped_rows} on {len(non_equity)} (Ticker, Company) pairs; {len(trades)} rows remain")
display(non_equity[["Ticker", "Company", "RowsInTrades", "Reason"]])

Non-equity rows dropped: 21 on 5 (Ticker, Company) pairs; 27628 rows remain

,Ticker,Company,RowsInTrades,Reason
0,BLK,BLACKROCK FDG INC,1,BlackRock Funding Inc. is BlackRock's debt-iss...
1,BLK,BLACKROCK FUNDING INC/DE,6,BlackRock Funding Inc. is BlackRock's debt-iss...
2,BP,BP CAPITAL MARKETS PLC,9,BP Capital Markets plc is BP's financing vehic...
3,SCGLY,SOCIETE GENERALE SA,4,Bought in the same weeks as the BP notes; trea...
4,CAST,CAST CORP,1,Issuer unidentifiable (FMP's CAST is FreeCast ...


### 5.5) FMP company names for the new tickers
Tickers that enter the file here are looked up at FMP exactly as in 4.3 and appended to `RawData/FMP_company_names.xlsx`; then `FMPCompanyName` is refreshed for every row. Where FMP does not know a very new symbol yet, the name from the review sheet (wrong-company rows) or from FMP's symbol-change record (renames) is used instead of "NA", and the row is listed.

In [34]:
fmp_lookup = pd.read_excel(FMP_LOOKUP_FILE, keep_default_na=False)
new_tickers = sorted(set(trades["Ticker"]) - set(fmp_lookup["Ticker"]))
print(f"Looking up {len(new_tickers)} tickers at FMP that the name file does not know: {new_tickers or 'none'}")
new_names = {t: lookup_company_name(t) for t in new_tickers}
fallback = dict(zip(wrong["CorrectedTicker"], wrong["CorrectedCompanyName"]))
for c in usable:
    fallback.setdefault(c["newSymbol"], (c.get("companyName") or "").replace(" Common Stock", "").strip())
filled = [t for t, n in new_names.items() if n == "NA" and fallback.get(t)]
for t in filled:
    new_names[t] = fallback[t]
if new_tickers:
    fmp_lookup = pd.concat([fmp_lookup, pd.DataFrame({"Ticker": new_tickers, "FMPCompanyName": [new_names[t] for t in new_tickers]})],
                           ignore_index=True).sort_values("Ticker").reset_index(drop=True)
    fmp_lookup.to_excel(FMP_LOOKUP_FILE, index=False)
trades["FMPCompanyName"] = trades["Ticker"].map(fmp_lookup.set_index("Ticker")["FMPCompanyName"]).fillna("NA")
entered = sorted(set(trades["Ticker"]) - tickers_at_import)
print(f"{FMP_LOOKUP_FILE}: {len(fmp_lookup)} tickers; names taken from the review sheet / symbol-change record instead of FMP search: {filled or 'none'}")
display(fmp_lookup[fmp_lookup["Ticker"].isin(entered)].reset_index(drop=True))

Looking up 0 tickers at FMP that the name file does not know: none
RawData\FMP_company_names.xlsx: 1677 tickers; names taken from the review sheet / symbol-change record instead of FMP search: none


,Ticker,FMPCompanyName
0,ACH,"Accendra Health, Inc."
1,ACHC,"Acadia Healthcare Company, Inc."
2,ADAM,"Adamas Trust, Inc."
3,CASK,"Sempreon, Inc."
4,CATY,Cathay General Bancorp
5,CFLT,"Confluent, Inc."
6,CRK,"Comstock Resources, Inc."
7,CSW,"CSW Industrials, Inc."
8,DCOM,"Dime Community Bancshares, Inc."
9,DVAX,Dynavax Technologies Corporation


### 5.6) Checks and export
Row count equals the count after section 4 minus the dropped non-equity rows; no row left on a pair from the `WrongCompanyAfterPrices` or `NonEquity` sheets; no row left on an applied old ticker; every corrected ticker present; no FMP company name with two tickers; every `Unpriceable` ticker a purchased ticker. Then `Trump_trades_2025_2026.xlsx` is written anew: `Trades` (the final file), `TradesBeforePriceReview` (the state after section 4, same columns), `TickerCompanyLookup`, `CompanyTickerGroups`; section 6 adds the purchase sheets and the change log.

In [35]:
applied_old = set(renames.loc[renames["Decision"] == "apply", "OldTicker"])
latest_of = dict(zip(renames.loc[renames["Decision"] == "apply", "OldTicker"], renames.loc[renames["Decision"] == "apply", "LatestTicker"]))
is_purchase = trades["Transaction"].str.lower().str.startswith("purchase")
checks = {
    "row count = rows after section 4 minus dropped non-equity rows": len(trades) == rows_before - dropped_rows,
    "no row left on a WrongCompanyAfterPrices pair": not any(((trades["Ticker"] == r.Ticker) & (trades["Company"] == r.Company)).any() for r in wrong.itertuples()),
    "no row left on a NonEquity pair": not any(((trades["Ticker"] == r.Ticker) & (trades["Company"] == r.Company)).any() for r in non_equity.itertuples()),
    "no row left on an applied old ticker": not trades["Ticker"].isin(applied_old).any(),
    "every corrected ticker present (after renames)": {latest_of.get(t, t) for t in wrong["CorrectedTicker"]} <= set(trades["Ticker"]),
    "every applied latest ticker present": set(renames.loc[renames["Decision"] == "apply", "LatestTicker"]) <= set(trades["Ticker"]),
    "no empty FMPCompanyName": trades["FMPCompanyName"].notna().all(),
    "every Unpriceable ticker is a purchased ticker": set(unpriceable["Ticker"]) <= set(trades.loc[is_purchase, "Ticker"]),
}
for k, v in checks.items():
    print(f"{'OK ' if v else 'FAIL'} {k}")
assert all(checks.values())

company_ticker_groups = (
    trades[trades["FMPCompanyName"].ne("NA")]
    .groupby("FMPCompanyName", as_index=False)
    .agg(Tickers=("Ticker", lambda v: " | ".join(sorted(set(v)))), TickerCount=("Ticker", "nunique"), TradeRowCount=("Ticker", "size"))
    .sort_values(["TickerCount", "FMPCompanyName"], ascending=[False, True]).reset_index(drop=True))
print(f"Companies with more than one ticker after the review: {len(company_ticker_groups[company_ticker_groups['TickerCount'] > 1])}")

final = trades.drop(columns=["TickerBefore"]).copy(); final["Traded"] = final["Traded"].dt.strftime("%Y-%m-%d")
before = trades_c7.copy(); before["Traded"] = pd.to_datetime(before["Traded"]).dt.strftime("%Y-%m-%d")
with pd.ExcelWriter(OUT_FILE, engine="openpyxl") as writer:          # written anew: the sheets of sections 1-4 are superseded
    final.to_excel(writer, sheet_name="Trades", index=False)
    before.to_excel(writer, sheet_name="TradesBeforePriceReview", index=False)
    final[["Ticker", "FMPCompanyName"]].drop_duplicates().sort_values("Ticker").to_excel(writer, sheet_name="TickerCompanyLookup", index=False)
    company_ticker_groups.to_excel(writer, sheet_name="CompanyTickerGroups", index=False)
Trump_trades_2025_2026 = final
print(f"Exported {OUT_FILE}: Trades {len(final)} rows, {final['Ticker'].nunique()} unique tickers | TradesBeforePriceReview {len(before)} rows, {before['Ticker'].nunique()} tickers")

OK  row count = rows after section 4 minus dropped non-equity rows
OK  no row left on a WrongCompanyAfterPrices pair
OK  no row left on a NonEquity pair
OK  no row left on an applied old ticker
OK  every corrected ticker present (after renames)
OK  every applied latest ticker present
OK  no empty FMPCompanyName
OK  every Unpriceable ticker is a purchased ticker


Companies with more than one ticker after the review: 0


Exported Trump_trades_2025_2026.xlsx: Trades 27628 rows, 1620 unique tickers | TradesBeforePriceReview 27649 rows, 1629 tickers


## 6) The purchase table for the price step

Step 3 (folder `3) Prices`) prices purchases only; sales stay in `Trades` for the optional checks. Three sheets are added to the output file:

* `Purchases`: the purchase rows of `Trades` in the five columns the price scripts read (`ticker`, `trade_date`, `Company`, `amount_low`, `amount_high`), one row per disclosed lot, minus the tickers of sheet `Unpriceable`; also written to `3) Prices/RawData/purchases.csv`, the input of the price scripts.
* `PurchasesMerged`: the same purchases with the lots of one company on one day added into one row (the cookbook's Step 2.3), with `amount_low`, `amount_mid` and `amount_high` summed and the number of lots kept; this is the table the hit count and the return test read.
* `PurchasesLeftOut`: the purchases of the `Unpriceable` tickers with the reason.

The `ChangeLog` sheet closes the file: one row per trade row whose ticker changed in section 5, that was dropped, or whose purchases are not priced, with the ticker before, the ticker after, the rule and the review sheet that decided it.

In [36]:
purch = (final[final["Transaction"].str.lower().str.startswith("purchase")]
         .rename(columns={"Ticker": "ticker", "Traded": "trade_date"})[["ticker", "trade_date", "Company", "amount_low", "amount_high"]]
         .sort_values(["trade_date", "ticker", "Company"]).reset_index(drop=True))
left_out_tickers = set(unpriceable["Ticker"])
left_out = purch[purch["ticker"].isin(left_out_tickers)].merge(unpriceable[["Ticker", "Reason"]].rename(columns={"Ticker": "ticker", "Reason": "reason"}), on="ticker", how="left")
purch = purch[~purch["ticker"].isin(left_out_tickers)].reset_index(drop=True)
for t, g in left_out.groupby("ticker"):
    changes_log.append(("left_out_of_purchases", t, "(not priced)", g["Company"].iloc[0], len(g)))
merged = (purch.assign(amount_mid=(purch["amount_low"] + purch["amount_high"]) / 2)
          .groupby(["ticker", "trade_date"], as_index=False)
          .agg(Company=("Company", lambda v: " | ".join(sorted(set(v)))), lots=("amount_low", "size"),
               amount_low=("amount_low", "sum"), amount_mid=("amount_mid", "sum"), amount_high=("amount_high", "sum"))
          .sort_values(["trade_date", "ticker"]).reset_index(drop=True))
PURCHASES_CSV.parent.mkdir(parents=True, exist_ok=True)
purch.to_csv(PURCHASES_CSV, index=False)
with pd.ExcelWriter(OUT_FILE, engine="openpyxl", mode="a", if_sheet_exists="replace") as writer:
    purch.to_excel(writer, sheet_name="Purchases", index=False)
    merged.to_excel(writer, sheet_name="PurchasesMerged", index=False)
    left_out.to_excel(writer, sheet_name="PurchasesLeftOut", index=False)
print(f"Purchases: {len(purch)} lots on {purch['ticker'].nunique()} tickers (sheet Purchases and {PURCHASES_CSV}) | "
      f"PurchasesMerged: {len(merged)} company-days ({len(purch) - len(merged)} lots added into another row) | "
      f"left out: {len(left_out)} purchases on {left_out['ticker'].nunique()} tickers")
display(left_out.groupby(["ticker", "Company", "reason"]).agg(purchases=("trade_date", "size"), first=("trade_date", "min"), last=("trade_date", "max")).reset_index())

Purchases: 18797 lots on 1594 tickers (sheet Purchases and ..\3) Prices\RawData\purchases.csv) | PurchasesMerged: 17724 company-days (1073 lots added into another row) | left out: 21 purchases on 5 tickers


,ticker,Company,reason,purchases,first,last
0,BRKL,BROOKLINE BANCORP INC DE,Brookline Bancorp merged into Berkshire Hills ...,4,2025-01-30,2025-05-29
1,NVRI,ENVIRI CORP,Enviri re-registered in 2026; FMP's NVRI serie...,6,2025-01-30,2025-05-22
2,PINC,PREMIER INC,Premier Inc. was taken private in 2025; FMP's ...,6,2025-01-30,2025-05-15
3,RNAM,AVIDITY BIOSCIENCES INC,"Avidity Biosciences traded as RNA, a symbol no...",1,2025-04-04,2025-04-04
4,WOLF,WOLFSPEED INC,Wolfspeed's old shares were cancelled in the C...,4,2025-01-30,2025-05-15


In [37]:
log = pd.DataFrame(changes_log, columns=["rule", "old_ticker", "new_ticker", "company", "rows"])
log = log[log["rows"] > 0].reset_index(drop=True)
re_tickered = log.loc[log["rule"].isin(["wrong_company", "latest_ticker"]), "rows"].sum()
print(f"Section 5 and 6 in numbers: rows re-tickered {re_tickered} of {rows_before} ({re_tickered / rows_before:.1%}): "
      f"{log.loc[log['rule'] == 'wrong_company', 'rows'].sum()} wrong company, {log.loc[log['rule'] == 'latest_ticker', 'rows'].sum()} latest ticker; "
      f"rows dropped as non-equity: {log.loc[log['rule'] == 'non_equity_dropped', 'rows'].sum()}; purchases left out of the price step: {log.loc[log['rule'] == 'left_out_of_purchases', 'rows'].sum()}; rows in Trades: {len(final)}")
display(log)

sheet_of = {"wrong_company": "WrongCompanyAfterPrices", "latest_ticker": "TickerRenames", "non_equity_dropped": "NonEquity", "left_out_of_purchases": "Unpriceable"}
wrong_pairs = wrong.set_index(["Ticker", "Company"])["CorrectedTicker"]
changed = trades[trades["Ticker"] != trades["TickerBefore"]].copy()
changed["rule"] = [("wrong_company" + (" + latest_ticker" if wrong_pairs.loc[(o, c)] != n else "")) if (o, c) in wrong_pairs.index else "latest_ticker"
                   for o, c, n in zip(changed["TickerBefore"], changed["Company"], changed["Ticker"])]
changed["status"] = "ticker changed"
dropped["rule"], dropped["status"] = "non_equity_dropped", "row dropped"
lo = trades[trades["Ticker"].isin(left_out_tickers) & is_purchase].copy()
lo["rule"], lo["status"] = "left_out_of_purchases", "row kept, purchase not priced"
change_log = pd.concat([changed, dropped, lo], ignore_index=True)
change_log["review_sheet"] = change_log["rule"].map(lambda r: " + ".join(sheet_of[x] for x in r.split(" + ")))
change_log["Traded"] = pd.to_datetime(change_log["Traded"]).dt.strftime("%Y-%m-%d")
change_log = (change_log.rename(columns={"TickerBefore": "ticker_before", "Ticker": "ticker_after"})
              [["status", "rule", "review_sheet", "ticker_before", "ticker_after", "Company", "Transaction", "Traded", "amount_low", "amount_high", "FMPCompanyName"]]
              .sort_values(["status", "ticker_before", "Traded"]).reset_index(drop=True))
with pd.ExcelWriter(OUT_FILE, engine="openpyxl", mode="a", if_sheet_exists="replace") as writer:
    change_log.to_excel(writer, sheet_name="ChangeLog", index=False)
left = sorted(tickers_at_import - set(final["Ticker"]))
print(f"ChangeLog: {len(change_log)} rows ({change_log['status'].value_counts().to_dict()})")
print(f"Tickers that left the universe ({len(left)}): {left}")
print(f"Tickers new to the universe ({len(entered)}): {entered}")
print(f"Sheets in {OUT_FILE}: {pd.ExcelFile(OUT_FILE).sheet_names}")

Section 5 and 6 in numbers: rows re-tickered 385 of 27649 (1.4%): 184 wrong company, 201 latest ticker; rows dropped as non-equity: 21; purchases left out of the price step: 21; rows in Trades: 27628


,rule,old_ticker,new_ticker,company,rows
0,wrong_company,NCL,NTRS,NORTHERN TR CORP,6
1,wrong_company,HSAI,HES,HESS CORP,6
2,wrong_company,MMED,NARI,INARI MED INC,5
3,wrong_company,CNDT,CFLT,CONFLUENT INC,2
4,wrong_company,AEVA,DVAX,DYNAVAX TECHNOLOGIES COR,11
5,wrong_company,BOX,JACK,JACK IN THE BOX INC,11
6,wrong_company,CBFV,PNC,PNC FINANCIAL SERVICES GROUP,6
7,wrong_company,CBFV,PNC,PNC FINANCIAL SERVICES GROUP I,2
8,wrong_company,COFS,DFS,DISCOVER FINANCIAL SERVICES,2
9,wrong_company,HCA,ACHC,ACADIA HEALTHCARE CO INC,4


ChangeLog: 423 rows ({'ticker changed': 381, 'row dropped': 21, 'row kept, purchase not priced': 21})
Tickers that left the universe (39): ['AEVA', 'AMBC', 'ASGN', 'ATGE', 'AXL', 'BHLB', 'BK', 'BP', 'CAST', 'CBFV', 'CNDT', 'COFS', 'CSWI', 'DCBG', 'EQR', 'FKWL', 'HONAV', 'HSAI', 'IAC', 'IBCP', 'LODE', 'MGPI', 'MMC', 'MMED', 'MODG', 'MSGS', 'NCL', 'NYMT', 'OBDE', 'OMI', 'PGRE', 'PSKY', 'PSTG', 'SCGLY', 'SCVL', 'SJW', 'TORO', 'VERX', 'VSNTV']
Tickers new to the universe (30): ['ACH', 'ACHC', 'ADAM', 'CASK', 'CATY', 'CFLT', 'CRK', 'CSW', 'DCOM', 'DVAX', 'FBNC', 'FFBC', 'HES', 'HONA', 'INDB', 'JACK', 'MSGK', 'NARI', 'NBTB', 'ONB', 'OSG', 'P', 'SHOE', 'SKYD', 'TTC', 'VBTX', 'VMRK', 'VSNT', 'WABC', 'WHR']
Sheets in Trump_trades_2025_2026.xlsx: ['Trades', 'TradesBeforePriceReview', 'TickerCompanyLookup', 'CompanyTickerGroups', 'Purchases', 'PurchasesMerged', 'PurchasesLeftOut', 'ChangeLog']


→ `Trump_trades_2025_2026.xlsx` is the trade file every later step reads: one ticker per company, the one the price provider serves, every change traceable to a review sheet, and the purchase table that Step 3 prices.